# Electronic and Embedding Responses to Torsional Coupling

## Setup

In [ ]:
from setup import *

import hashlib
from pyscf import gto, dft
from rdkit.Chem import rdMolTransforms

## Compute Torsional Coupling

In [ ]:
theta1, theta2 = 60.0, 180.0
# (analogue B, parent A, group). Every SMILES is written end-C-C-end, so the torsion is atoms 0-1-2-3. Each molecule has
# one SMILES: Y-C-C-X and X-C-C-Y are the same molecule, and the parent CH3-C-C-Y is the same molecule as the control
# CH3-C-C-X with X = Y, so it is written CCC{Y} in both roles (one geometry and one DFT energy per molecule).
groups = ["O", "N", "S", "F", "Cl", "Br"]  # OH, NH2, SH, F, Cl, Br as SMILES atoms (hydrogens implicit)
torsion_pairs = []
for i, Y in enumerate(groups):
    for X in groups[i:]:
        B = f"{Y}CC{X}"
        torsion_pairs.append((B, f"CCC{Y}", "coupled"))  # X replaced by CH3
        if X != Y:
            torsion_pairs.append((B, f"CCC{X}", "coupled"))  # Y replaced by CH3
for X in groups:
    torsion_pairs.append((f"CCC{X}", "CCCC", "control"))  # X's partner is a CH3; the parent is butane
model_paths = {name: CHECKPOINTS[name] for name in ("nopre", "NT", "ANT")}
representations = ["Pooled AIMNet2"] + list(model_paths)

# ---- geometry of every state: the lowest-MMFF-energy of 20 conformers, each relaxed with the torsion held at theta ----
def lowest_conformer(smiles, theta, n_starts=20):
    mol = Chem.AddHs(Chem.MolFromSmiles(smiles))
    candidates = []
    for conf_id in AllChem.EmbedMultipleConfs(mol, n_starts, randomSeed=42):  # not 0: randomSeed=0 repeats one conformer
        conformer = Chem.Mol(mol, confId=conf_id)
        rdMolTransforms.SetDihedralDeg(conformer.GetConformer(), 0, 1, 2, 3, theta)
        properties = AllChem.MMFFGetMoleculeProperties(conformer)
        force_field = AllChem.MMFFGetMoleculeForceField(conformer, properties)
        force_field.MMFFAddTorsionConstraint(0, 1, 2, 3, False, theta, theta, 1e4)
        force_field.Minimize(maxIts=5000)
        rdMolTransforms.SetDihedralDeg(conformer.GetConformer(), 0, 1, 2, 3, theta)  # the constraint is soft; make it exact
        energy = AllChem.MMFFGetMoleculeForceField(conformer, properties).CalcEnergy()  # without the constraint
        candidates.append((energy, conformer))
    return min(candidates, key=lambda candidate: candidate[0])[1]

molecules = sorted({B for B, A, group in torsion_pairs} | {A for B, A, group in torsion_pairs})
states = [(smiles, theta) for smiles in molecules for theta in (theta1, theta2)]
geometry = {state: lowest_conformer(*state) for state in states}

# ---- DFT energy E and embeddings z of every state, each as {(smiles, theta): value} ----
known_energies = pd.read_csv(DFT_CACHE, index_col="key").energy.to_dict()
def dft_energy(mol):
    """B3LYP-D3(BJ)/def2-TZVP single-point energy in kcal/mol, reused from the saved energies if this geometry was done before."""
    coordinates = np.round(mol.GetConformer().GetPositions(), 4)
    key = hashlib.sha1(Chem.MolToSmiles(mol).encode() + coordinates.tobytes()).hexdigest()  # identifies the geometry
    if key not in known_energies:
        atoms = [(atom.GetSymbol(), xyz) for atom, xyz in zip(mol.GetAtoms(), mol.GetConformer().GetPositions())]
        # max_memory (MB): pyscf counts this whole process (models, torch) against it, and at its 4 GB default it moves
        # the density-fitting integrals to disk, which makes every calculation ~14x slower
        molecule = gto.M(atom=atoms, basis="def2-tzvp", unit="Angstrom", verbose=0, max_memory=16000)
        calculation = dft.RKS(molecule, xc="b3lyp").density_fit()
        calculation.disp = "d3bj"
        energy = calculation.kernel()
        assert calculation.converged
        known_energies[key] = energy * 627.509474  # hartree -> kcal/mol
        pd.Series(known_energies, name="energy").rename_axis("key").to_csv(DFT_CACHE)  # saved after every new energy
    return known_energies[key]

E = {state: dft_energy(geometry[state]) for state in tqdm(states, desc="DFT")}
embeddings = encode_molecules_with_models(model_paths, [geometry[state] for state in states], DEVICE)
z = {rep: dict(zip(states, embeddings[rep])) for rep in representations}

# ---- the double difference of every pair ----
def double_difference(P, A, B, theta=theta2):
    """[P(B,θ) - P(A,θ)] - [P(B,θ1) - P(A,θ1)] for a property P = {(smiles, theta): value}; θ is θ2 unless given."""
    return (P[B, theta] - P[A, theta]) - (P[B, theta1] - P[A, theta1])

rows = []
for B, A, group in torsion_pairs:
    row = {"B": B, "A": A, "group": group, "ΔΔE (kcal/mol)": double_difference(E, A, B)}
    for rep in representations:
        row[rep] = np.linalg.norm(double_difference(z[rep], A, B))  # ‖Δ²z‖
    rows.append(row)
pairs = pd.DataFrame(rows)

# ---- results: does ‖Δ²z‖ rank pairs like |ΔΔE|, and do coupled pairs stand out from the controls as much? ----
coupling = pairs["ΔΔE (kcal/mol)"].abs()
is_coupled = pairs.group == "coupled"
def coupled_to_control(values):
    return values[is_coupled].mean() / values[~is_coupled].mean()
rho = {rep: spearmanr(coupling, pairs[rep])[0] for rep in representations}
coupling_ratios = {rep: coupled_to_control(pairs[rep]) for rep in representations}
dft_coupled_control_ratio = coupled_to_control(coupling)

## Figure 9: Electronic and Embedding Torsional Coupling

In [ ]:
colors = {"coupled": plt.get_cmap("Spectral")(0.05), "control": plt.get_cmap("Spectral")(0.88)}
labels = {"coupled": "Coupled Pairs", "control": "Control Pairs (CH$_3$ partner)"}
fig, axes = plt.subplots(2, 2, figsize=(7.1 * cm, 6.8 * cm), sharex=True)  # x (DFT |ΔΔE|) is the same in every panel
for ax, rep in zip(axes.flat, representations):
    for group in ["coupled", "control"]:
        in_group = pairs.group == group
        ax.scatter(coupling[in_group], pairs[rep][in_group], s=5, color=colors[group], linewidths=0, label=labels[group])
    ax.text(0.04, 0.97, f"ρ = {rho[rep]:.2f}", transform=ax.transAxes, fontsize=4.5, va="top")
    ax.set_xlim(0, coupling.max() * 1.1)
    ax.set_ylim(0, pairs[rep].max() * 1.2)
    ax.set_box_aspect(1)  # square panel
    ax.set_title(MODEL_TITLES.get(rep, rep), fontsize=6, pad=1)
    ax.locator_params(axis="y", nbins=6, steps=[1, 2, 5, 10])  # tick steps like 0.005 or 0.02, never 0.0025: short tick labels
    ax.tick_params(labelsize=4, length=2, width=0.4, pad=1)
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_linewidth(0.4)
# edges of the panel block: equal left and right margins, so the four panels are centered in the figure
left, right, top, bottom = 0.106, 0.894, 0.956, 0.147
# one shared label per axis, as in the RMSD heatmap figure; the x label and legend are centered under the panels
fig.text((left + right) / 2, 0.085, "DFT |ΔΔE| (kcal/mol)", fontsize=6, ha="center", va="center")
fig.text(left - 0.081, (top + bottom) / 2, "‖Δ²z‖", fontsize=6, rotation=90, ha="center", va="center")
fig.legend(*axes[0, 0].get_legend_handles_labels(), frameon=False, fontsize=5, loc="lower center", bbox_to_anchor=((left + right) / 2, 0),
           ncol=2, handletextpad=0.3, columnspacing=1.2)
fig.subplots_adjust(left=left, right=right, top=top, bottom=bottom, hspace=0.136, wspace=0.175)
fig.savefig(OUTPUT_DIR / "torsion_coupling_methyl.png", dpi=600)
plt.show()

## Four-State Pair Measurements

In [ ]:
display(pairs.rename(columns=MODEL_TITLES).round(3))

## Coupling Correlations and Group Ratios

In [ ]:
display(pd.DataFrame({
    "Spearman rho": rho,
    "Coupled/control mean ratio": coupling_ratios,
}).rename(index=MODEL_TITLES).round(3))

## DFT Coupled-to-Control Ratio

In [ ]:
display(pd.DataFrame([{"Coupled/control mean ratio": dft_coupled_control_ratio}]))